# 3D Deformation Field Correction — Test Cases

This notebook tests the 3D extension of the iterative SLSQP algorithm for
correcting negative Jacobian determinants in volumetric deformation fields.

Deformation fields have shape `(3, D, H, W)` with channels `[dz, dy, dx]`.
The 3D Jacobian determinant is the determinant of the 3×3 deformation
gradient tensor `det(I + ∇u)`, computed via `np.gradient` central differences.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from modules.dvfopt3d import (
    generate_random_dvf_3d,
    scale_dvf_3d,
    jacobian_det3D,
    iterative_3d,
)

## Sanity Check — Identity Deformation

A zero displacement field should have Jacobian determinant = 1.0 everywhere.

In [ ]:
zero_dvf = np.zeros((3, 5, 5, 5))
jdet_zero = jacobian_det3D(zero_dvf)
print(f"Shape: {jdet_zero.shape}")
print(f"Min: {jdet_zero.min():.6f}  Max: {jdet_zero.max():.6f}")
assert np.allclose(jdet_zero, 1.0), "Identity Jdet should be 1.0 everywhere"
print("PASS: Identity deformation has Jdet = 1.0 everywhere")

## Helper — Summarise Jacobian Stats

In [ ]:
def jdet_summary(dvf, label="DVF"):
    """Print Jacobian determinant summary for a (3, D, H, W) field."""
    jdet = jacobian_det3D(dvf)
    neg = int((jdet <= 0).sum())
    total = jdet.size
    print(f"{label}:  shape={dvf.shape[1:]}  "
          f"neg={neg}/{total}  min={jdet.min():.6f}  max={jdet.max():.6f}")
    return jdet

---
## Test Case 1 — Small Cubic Grid (5×5×5)

Random DVF with moderate magnitude. Small enough for fast SLSQP convergence.

In [ ]:
dvf_1 = generate_random_dvf_3d((3, 5, 5, 5), max_magnitude=2.0, seed=42)
jdet_before_1 = jdet_summary(dvf_1, "Case 1 — before")

phi_1 = iterative_3d(dvf_1, verbose=1)

jdet_after_1 = jdet_summary(phi_1, "Case 1 — after")
assert (jdet_after_1 > 0).all(), "All Jdet must be positive after correction"

---
## Test Case 2 — Non-Cubic Grid (4×6×5)

Tests rectangular sub-volume handling and full-grid fallback.

In [ ]:
dvf_2 = generate_random_dvf_3d((3, 4, 6, 5), max_magnitude=1.5, seed=123)
jdet_before_2 = jdet_summary(dvf_2, "Case 2 — before")

phi_2 = iterative_3d(dvf_2, verbose=1)

jdet_after_2 = jdet_summary(phi_2, "Case 2 — after")
assert (jdet_after_2 > 0).all(), "All Jdet must be positive after correction"

---
## Test Case 3 — Upscaled Smooth DVF (3×3×3 → 8×8×8)

Generate a coarse random field and upscale with trilinear interpolation.
This produces a smoother field with fewer but larger negative-Jdet regions.

In [ ]:
dvf_coarse = generate_random_dvf_3d((3, 3, 3, 3), max_magnitude=4.0, seed=7)
dvf_3 = scale_dvf_3d(dvf_coarse, (8, 8, 8))
jdet_before_3 = jdet_summary(dvf_3, "Case 3 — before")

phi_3 = iterative_3d(dvf_3, verbose=1)

jdet_after_3 = jdet_summary(phi_3, "Case 3 — after")
assert (jdet_after_3 > 0).all(), "All Jdet must be positive after correction"

---
## Visualisation — Jacobian Slices Before vs. After

Show 2D Jacobian determinant heatmaps for each z-slice of the last test case.

In [ ]:
def plot_jdet_slices(dvf_before, dvf_after, title_prefix=""):
    """Plot before/after Jdet heatmaps for every z-slice."""
    jdet_b = jacobian_det3D(dvf_before)
    jdet_a = jacobian_det3D(dvf_after)
    D = jdet_b.shape[0]
    vmin = min(jdet_b.min(), jdet_a.min())
    vmax = max(jdet_b.max(), jdet_a.max())

    fig, axes = plt.subplots(2, D, figsize=(3 * D, 6))
    if D == 1:
        axes = axes[:, np.newaxis]
    for z in range(D):
        ax_b = axes[0, z]
        im = ax_b.imshow(jdet_b[z], vmin=vmin, vmax=vmax, cmap='RdBu_r')
        ax_b.set_title(f'Before z={z}')
        ax_b.axis('off')

        ax_a = axes[1, z]
        ax_a.imshow(jdet_a[z], vmin=vmin, vmax=vmax, cmap='RdBu_r')
        ax_a.set_title(f'After z={z}')
        ax_a.axis('off')

    fig.colorbar(im, ax=axes, shrink=0.6, label='Jdet')
    fig.suptitle(f'{title_prefix} Jacobian Determinant — Before vs After', fontsize=14)
    plt.tight_layout()
    plt.show()


plot_jdet_slices(dvf_3, phi_3, title_prefix="Case 3:")

In [ ]:
plot_jdet_slices(dvf_1, phi_1, title_prefix="Case 1:")

---
## Summary

All test cases should report **0 negative Jacobian determinants** after correction,
with `min_jdet >= threshold (0.01)`.